# Lab 15-02: ACA Fine-Tuning Job

This notebook:
1. Reads infra configuration from `FINETUNE_*` environment variables
2. Provisions ACA environment + storage (idempotent via `azure_infra.py`)
3. Submits the Olive LoRA fine-tuning job to an A100 GPU on Azure Container Apps
4. Monitors job progress until completion

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
repo_root = Path.cwd().parents[0]
load_dotenv(repo_root / '.env', override=True)

In [ ]:
import os
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from azure_infra import provision_infrastructure, submit_finetune_job, monitor_job

## 4. Fine-Tune on Azure Container Apps (Serverless GPU)

Submit a fine-tuning job to Azure Container Apps with an **NVIDIA A100 GPU** using **Microsoft Olive**.

In [ ]:
# Configuration from environment variables
RESOURCE_GROUP = os.getenv("FINETUNE_RESOURCE_GROUP")
STORAGE_ACCOUNT = os.getenv("FINETUNE_STORAGE_ACCOUNT")
ACA_ENV = os.getenv("FINETUNE_ACA_ENVIRONMENT")
BASE_MODEL_ID = "microsoft/Phi-4-mini-instruct"
LOCATION = "swedencentral"  # ACA GPU NC24-A100 only available in Sweden Central
CONTAINER_NAME = "ft"
JOB_NAME = "iss-ft-job"

print(f"Resource Group:  {RESOURCE_GROUP}")
print(f"Storage Account: {STORAGE_ACCOUNT}")
print(f"ACA Environment: {ACA_ENV}")
print(f"Base Model:      {BASE_MODEL_ID}")

In [ ]:
print(f"Training Config: {RESOURCE_GROUP} | {LOCATION}")

# 1. Provision Infrastructure
env_id = provision_infrastructure(RESOURCE_GROUP, LOCATION, STORAGE_ACCOUNT, CONTAINER_NAME, ACA_ENV, "data/train.jsonl")

# 2. Submit Job
submit_finetune_job(
    JOB_NAME, RESOURCE_GROUP, env_id, STORAGE_ACCOUNT, CONTAINER_NAME, 
    base_model=BASE_MODEL_ID, location=LOCATION
)

# 3. Monitor (this takes ~15-20 mins)
success = monitor_job(JOB_NAME, RESOURCE_GROUP)


In [ ]:
if success:
    print("✅ Fine-tuning completed successfully!")
    print("Adapter saved to blob storage. Will evaluate on ACA next.")
else:
    print("❌ Training failed. Check Azure Portal for logs.")